### <center> ???????????? ??????? ? ?????? ?4

1. ??????? ????? ?????? ????????????? ???????? ?????? https://www.kaggle.com/davinwijaya/customer-retention
2. ??? ???? conversion - ??? ??????? ??????????, ? offer - ????????????. ????????????? ???? (conversion -> target, offer -> treatment) ? ???????? ???? treatment ? ????????? ???? (1 ??? 0, ?.? ???? ?????-?? ??????????? ??? ???) - ???????? No Offer ???????? ?????????? ????????????, ? ??? ????????? - ???????.
3. ??????? ????????? ?????? ?????? ?? ????????????? ? ???????? ???????
4. ??????? feature engineering ?? ???? ?????????? (??????????? ??????? ?????? ???????)
5. ???????? uplift-????????????? 3 ?????????: ???? ?????? ? ????????? ???????????? (S learner), ?????? ? ?????????????? ??????? (????????????? ??????? ?. 2. 1) ? ??????? ? ????? ???????????? ????????
6. ? ????? ??????? ?????? ??????? ????????? ?????? uplift@10%, uplift@20% ???? 3 ???????
7. ????????? ?????? UpliftTreeClassifier ? ?????????? ??????? ??????? ?????????? ??????
8. (???????????) ??? ?????? S learner (?????? ? ?????????????? ????????? ????????????) ????????? ??????????? ??????? (????????? - ???? conversion) ?? ???????? uplift: 1) ??????? ??????? ? ???????? uplift ??? ???????? ??????? 2) ????????????? ???????? ??????? ?? uplift ?? ???????? 3) ??????? ?? ?????? (pandas qcut ??? ? ??????) 4) ??? ??????? ?????? ????????? ??????? conversion
9. (???????????) ????????? ?????? UpliftRandomForestClassifier ? ?????????? ??????? ??????? ?????????? ??????

In [ ]:
import pandas as pd

**1. ??????? ????? ?????? ????????????? ????????**

In [ ]:
df = pd.read_csv('data.csv')
df

**2. ????????????? ???? (conversion -> target, offer -> treatment) ? ???????? ???? treatment ? ????????? ???? (1 ??? 0, ?.? ???? ?????-?? ??????????? ??? ???) - ???????? No Offer ???????? ?????????? ????????????, ? ??? ????????? - ???????.**

In [ ]:
df.rename(columns={'conversion': 'target', 'offer': 'treatment'}, inplace=True)
df

In [ ]:
df.info()


In [ ]:
df.treatment.unique()

In [ ]:
treatment_dict={'Buy One Get One':1, 'No Offer':0, 'Discount':1}
df.treatment=df.treatment.map(treatment_dict)

In [ ]:
df.treatment.value_counts()

**4. ??????? feature engineering ?? ???? ?????????? (??????????? ??????? ?????? ???????)**

?????????? ?????????????? ????????:

In [ ]:
df.zip_code.unique()

In [ ]:
df.channel.unique()

????????? ???????????? ???????????? CatBoost ????? ???????? ??? ???????? ??? ????...

In [ ]:
cat_features=['zip_code','channel']

????? ????????????? ???????? ???????? 'hist'

**3. ??????? ????????? ?????? ?????? ?? ????????????? ? ???????? ???????**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
features = ['recency', 'history', 'used_discount', 'used_bogo', 'zip_code','is_referral', 'channel']
treatment = 'treatment'
target = 'target'

indices_train, indices_test = train_test_split(df.index, test_size=0.3, random_state=0)

In [ ]:
X_train = df.loc[indices_train, features]
y_train = df.loc[indices_train, target]
treat_train = df.loc[indices_train, treatment]
X_test = df.loc[indices_test, features]
y_test = df.loc[indices_test, target]
treat_test = df.loc[indices_test, treatment]
models_results = {
    'approach': [],
    'uplift@10%': [],
    'uplift@20%': []
}

**5. ???????? uplift-????????????? 3 ?????????: ???? ?????? ? ????????? ???????????? (S learner), ?????? ? ?????????????? ??????? (????????????? ??????? ?. 2. 1) ? ??????? ? ????? ???????????? ????????**

#### 1 ??????

In [ ]:
from sklift.metrics import uplift_at_k
from sklift.viz import plot_uplift_preds
from sklift.models import SoloModel
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score, RandomizedSearchCV
import numpy as np

In [ ]:
parameters_cb = {
     'n_estimators': [100, 150, 200, 600, 1000],
     'max_depth': np.arange(2, 10, 2),
 }
clf = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        cat_features=cat_features, random_state=0, silent=True
    ),
    param_distributions=parameters_cb,
    scoring='roc_auc',
    cv=5
)
clf.fit(X_train, y_train)
clf.best_params_

In [ ]:
sm = SoloModel(CatBoostClassifier(n_estimators=600, max_depth=2, cat_features=cat_features, random_state=0, silent=True))
sm = sm.fit(X_train, y_train, treat_train)

uplift_sm = sm.predict(X_test)
uplift_sm

In [ ]:
sm_score_10 = uplift_at_k(y_true=y_test, uplift=uplift_sm, treatment=treat_test, strategy='by_group', k=0.1)
sm_score_10

In [ ]:
sm_score_20 = uplift_at_k(y_true=y_test, uplift=uplift_sm, treatment=treat_test, strategy='by_group', k=0.2)
sm_score_20

In [ ]:
models_results['approach'].append('SoloModel')
models_results['uplift@10%'].append(sm_score_10)
models_results['uplift@20%'].append(sm_score_20)

In [ ]:
# ??????? ???????? ??????????? ?????????? ???????? ???????? ??? ?????????????? ??? ??????? ???????
sm_trmnt_preds = sm.trmnt_preds_
# ? ???????? ??????????? ?????????? ???????? ???????? ??? ?????????????? ??? ??????? ???????
sm_ctrl_preds = sm.ctrl_preds_

# ???????? ????????????? ???????????? ? ?? ???????? (uplift)
plot_uplift_preds(trmnt_preds=sm_trmnt_preds, ctrl_preds=sm_ctrl_preds)

In [ ]:
# ? ??? ?? ????????? ????? ?????????? ? ????????? ??????.
# ????????, ????? ????????? ???????? ?????????:
sm_fi = pd.DataFrame({
    'feature_name': sm.estimator.feature_names_,
    'feature_score': sm.estimator.feature_importances_
}).sort_values('feature_score', ascending=False).reset_index(drop=True)

sm_fi

#### ????????????? ???????

In [ ]:
from sklift.models import ClassTransformation


ct = ClassTransformation(CatBoostClassifier(n_estimators=600, max_depth=2, cat_features=cat_features, random_state=0, silent=True))
ct = ct.fit(X_train, y_train, treat_train)

uplift_ct = ct.predict(X_test)

ct_score_10 = uplift_at_k(y_true=y_test, uplift=uplift_ct, treatment=treat_test, strategy='by_group', k=0.1)
ct_score_20 = uplift_at_k(y_true=y_test, uplift=uplift_ct, treatment=treat_test, strategy='by_group', k=0.2)

models_results['approach'].append('ClassTransformation')
models_results['uplift@10%'].append(ct_score_10)
models_results['uplift@20%'].append(ct_score_20)

In [ ]:
ct_score_10

In [ ]:
ct_score_20

#### ???????? ?????????

In [ ]:
ct_fi = pd.DataFrame({
    'feature_name': ct.estimator.feature_names_,
    'feature_score': ct.estimator.feature_importances_
}).sort_values('feature_score', ascending=False).reset_index(drop=True)

ct_fi

#### 2 ??????

In [ ]:
from sklift.models import TwoModels


tm = TwoModels(
    estimator_trmnt=CatBoostClassifier(n_estimators=600, max_depth=2, cat_features=cat_features, random_state=0, silent=True), 
    estimator_ctrl=CatBoostClassifier(n_estimators=600, max_depth=2, cat_features=cat_features, random_state=0, silent=True), 
    method='vanilla'
)
tm = tm.fit(
    X_train, y_train, treat_train)

uplift_tm = tm.predict(X_test)

tm_score_10 = uplift_at_k(y_true=y_test, uplift=uplift_tm, treatment=treat_test, strategy='by_group', k=0.1)
tm_score_20 = uplift_at_k(y_true=y_test, uplift=uplift_tm, treatment=treat_test, strategy='by_group', k=0.2)

models_results['approach'].append('TwoModels')
models_results['uplift@10%'].append(tm_score_10)
models_results['uplift@20%'].append(tm_score_20)

plot_uplift_preds(trmnt_preds=tm.trmnt_preds_, ctrl_preds=tm.ctrl_preds_)

In [ ]:
tm_score_10

In [ ]:
tm_score_20

#### ???????? ????????? ??? ?????? ??????

In [ ]:
tm_fi_trmnt = pd.DataFrame({
    'feature_name': tm.estimator_trmnt.feature_names_,
    'feature_score': tm.estimator_trmnt.feature_importances_
}).sort_values('feature_score', ascending=False).reset_index(drop=True)

tm_fi_trmnt

#### ???????? ????????? ??? ??????????? ??????

In [ ]:
tm_fi_ctrl = pd.DataFrame({
    'feature_name': tm.estimator_ctrl.feature_names_,
    'feature_score': tm.estimator_ctrl.feature_importances_
}).sort_values('feature_score', ascending=False).reset_index(drop=True)

tm_fi_ctrl

**6.??????? ?????? ??????? ????????? ?????? uplift@10%, uplift@20% ???? 3 ???????**

In [ ]:
pd.DataFrame(data=models_results).sort_values('uplift@10%', ascending=False)